# LVLM Training Visualization

This notebook provides tools to monitor and visualize training progress in real-time.

**Features:**
- Load and display training logs
- Plot loss curves (training vs validation)
- Track metrics across epochs
- Analyze checkpoint quality
- Compare different training runs

In [ ]:
# Import libraries
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# Set paths
exp_dir = Path('../experiments')
checkpoint_dir = exp_dir / 'checkpoints'
results_dir = exp_dir / 'results'

print(f"Experiments directory: {exp_dir}")
print(f"Checkpoint directory: {checkpoint_dir}")
print(f"Results directory: {results_dir}")

## 1. Generate Synthetic Training Data (for Demo)

In [ ]:
# Create synthetic training history for demonstration
def generate_synthetic_training_history():
    """Generate realistic synthetic training data for demo."""
    epochs = 20
    
    # Synthetic loss curves
    train_loss = 3.0 * np.exp(-np.arange(epochs) / 5) + 0.5 + np.random.normal(0, 0.1, epochs)
    val_loss = 3.0 * np.exp(-np.arange(epochs) / 5) + 0.6 + np.random.normal(0, 0.15, epochs)
    
    # Accuracy improves over time
    accuracy = (1 - np.exp(-np.arange(epochs) / 8)) * 0.7 + np.random.normal(0, 0.01, epochs)
    accuracy = np.clip(accuracy, 0, 1)
    
    # Temporal IoU improves
    temporal_iou = (1 - np.exp(-np.arange(epochs) / 6)) * 0.6 + np.random.normal(0, 0.02, epochs)
    temporal_iou = np.clip(temporal_iou, 0, 1)
    
    # Learning rate schedule
    lr = 1e-4 * (1 + np.cos(np.pi * np.arange(epochs) / epochs)) / 2
    
    history = {
        'epoch': list(range(1, epochs + 1)),
        'train_loss': train_loss.tolist(),
        'val_loss': val_loss.tolist(),
        'accuracy': accuracy.tolist(),
        'temporal_iou': temporal_iou.tolist(),
        'learning_rate': lr.tolist(),
    }
    
    return history

# Load training history
history_file = results_dir / 'training_history.json'
if history_file.exists():
    with open(history_file) as f:
        history = json.load(f)
    print(f"Loaded training history from {history_file}")
else:
    print("No training history found. Generating synthetic data for demonstration...")
    history = generate_synthetic_training_history()
    
    # Save for reference
    results_dir.mkdir(parents=True, exist_ok=True)
    with open(history_file, 'w') as f:
        json.dump(history, f, indent=2)

df_history = pd.DataFrame(history)
print(f"\nTraining data (first 5 epochs):")
print(df_history.head())

## 2. Loss Curves Analysis

In [ ]:
# Plot loss curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Training vs Validation Loss
axes[0, 0].plot(df_history['epoch'], df_history['train_loss'], marker='o', label='Training Loss', linewidth=2)
axes[0, 0].plot(df_history['epoch'], df_history['val_loss'], marker='s', label='Validation Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training vs Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Best epoch
best_epoch = df_history['val_loss'].idxmin()
best_val_loss = df_history.loc[best_epoch, 'val_loss']
axes[0, 0].axvline(x=best_epoch + 1, color='red', linestyle='--', alpha=0.5, label=f'Best (Epoch {best_epoch + 1})')
axes[0, 0].legend()

# Loss difference (overfit indicator)
loss_diff = np.array(df_history['val_loss']) - np.array(df_history['train_loss'])
axes[0, 1].plot(df_history['epoch'], loss_diff, marker='o', color='orange', linewidth=2)
axes[0, 1].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Val Loss - Train Loss')
axes[0, 1].set_title('Overfitting Indicator')
axes[0, 1].fill_between(df_history['epoch'], 0, loss_diff, alpha=0.3, color='orange')
axes[0, 1].grid(True, alpha=0.3)

# Accuracy and Temporal IoU
axes[1, 0].plot(df_history['epoch'], df_history['accuracy'], marker='o', label='Accuracy', linewidth=2, color='green')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].set_title('Model Accuracy Across Epochs')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Temporal IoU
axes[1, 1].plot(df_history['epoch'], df_history['temporal_iou'], marker='o', label='Temporal IoU', linewidth=2, color='purple')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Temporal IoU')
axes[1, 1].set_title('Temporal Grounding Quality')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best epoch: {best_epoch + 1}")
print(f"Best validation loss: {best_val_loss:.4f}")

## 3. Learning Rate Schedule

In [ ]:
# Plot learning rate decay
fig, ax = plt.subplots(1, 1, figsize=(12, 5))

ax.plot(df_history['epoch'], df_history['learning_rate'], marker='o', linewidth=2.5, markersize=8, color='steelblue')
ax.fill_between(df_history['epoch'], 0, df_history['learning_rate'], alpha=0.3)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Learning Rate', fontsize=12)
ax.set_title('Learning Rate Schedule (Cosine Annealing with Warmup)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

plt.tight_layout()
plt.show()

print(f"Initial learning rate: {df_history['learning_rate'].iloc[0]:.2e}")
print(f"Minimum learning rate: {df_history['learning_rate'].min():.2e}")
print(f"Final learning rate: {df_history['learning_rate'].iloc[-1]:.2e}")

## 4. Training Statistics

In [ ]:
# Compute statistics
print("\n=== Training Statistics ===")
print(f"\nLoss:")
print(f"  Train Loss - Final: {df_history['train_loss'].iloc[-1]:.4f}")
print(f"  Train Loss - Improvement: {df_history['train_loss'].iloc[0] - df_history['train_loss'].iloc[-1]:.4f}")
print(f"  Val Loss - Final: {df_history['val_loss'].iloc[-1]:.4f}")
print(f"  Val Loss - Best: {df_history['val_loss'].min():.4f} (Epoch {df_history['val_loss'].idxmin() + 1})")

print(f"\nAccuracy:")
print(f"  Initial: {df_history['accuracy'].iloc[0]:.4f}")
print(f"  Final: {df_history['accuracy'].iloc[-1]:.4f}")
print(f"  Best: {df_history['accuracy'].max():.4f}")
print(f"  Improvement: {df_history['accuracy'].iloc[-1] - df_history['accuracy'].iloc[0]:.4f}")

print(f"\nTemporal IoU:")
print(f"  Initial: {df_history['temporal_iou'].iloc[0]:.4f}")
print(f"  Final: {df_history['temporal_iou'].iloc[-1]:.4f}")
print(f"  Best: {df_history['temporal_iou'].max():.4f}")
print(f"  Improvement: {df_history['temporal_iou'].iloc[-1] - df_history['temporal_iou'].iloc[0]:.4f}")

print(f"\nOverfitting Analysis:")
avg_overfit = (np.array(df_history['val_loss']) - np.array(df_history['train_loss'])).mean()
print(f"  Avg overfitting: {avg_overfit:.4f}")
if avg_overfit > 0.2:
    print("  → Significant overfitting detected. Consider: more data, regularization, or early stopping")
else:
    print("  → Acceptable training/validation gap")

## 5. Checkpoint Quality

In [ ]:
# Analyze checkpoints
if checkpoint_dir.exists():
    checkpoint_files = sorted(checkpoint_dir.glob('checkpoint_epoch_*.pt'))
    print(f"Found {len(checkpoint_files)} checkpoints")
    
    # Get checkpoint sizes
    sizes = [f.stat().st_size / (1024**2) for f in checkpoint_files]  # MB
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Checkpoint sizes
    epochs = [int(f.stem.split('_')[-1]) for f in checkpoint_files]
    axes[0].bar(epochs, sizes, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Checkpoint Size (MB)')
    axes[0].set_title('Model Checkpoint Sizes')
    axes[0].grid(True, axis='y', alpha=0.3)
    
    # Metrics progress
    metrics_by_checkpoint = []
    for epoch in epochs:
        if epoch - 1 < len(df_history):
            metrics_by_checkpoint.append({
                'epoch': epoch,
                'accuracy': df_history.loc[epoch - 1, 'accuracy'],
                'temporal_iou': df_history.loc[epoch - 1, 'temporal_iou']
            })
    
    if metrics_by_checkpoint:
        df_metrics = pd.DataFrame(metrics_by_checkpoint)
        axes[1].plot(df_metrics['epoch'], df_metrics['accuracy'], marker='o', label='Accuracy', linewidth=2)
        axes[1].plot(df_metrics['epoch'], df_metrics['temporal_iou'], marker='s', label='Temporal IoU', linewidth=2)
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Metric Value')
        axes[1].set_title('Metrics at Each Checkpoint')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Total checkpoint storage: {sum(sizes):.1f} MB")
else:
    print("No checkpoints found. Train the model first.")

## 6. Training Summary Report

In [ ]:
# Generate summary report
summary_report = f"""
╔════════════════════════════════════════════════════════════════╗
║              LVLM TRAINING SUMMARY REPORT                      ║
╚════════════════════════════════════════════════════════════════╝

📊 OVERALL PERFORMANCE
  • Total Epochs: {len(df_history)}
  • Best Epoch: {best_epoch + 1} (Val Loss: {best_val_loss:.4f})
  • Final Accuracy: {df_history['accuracy'].iloc[-1]:.2%}
  • Final Temporal IoU: {df_history['temporal_iou'].iloc[-1]:.4f}

📈 LOSS PROGRESSION
  • Initial Train Loss: {df_history['train_loss'].iloc[0]:.4f}
  • Final Train Loss: {df_history['train_loss'].iloc[-1]:.4f}
  • Loss Improvement: {(df_history['train_loss'].iloc[0] - df_history['train_loss'].iloc[-1]) / df_history['train_loss'].iloc[0] * 100:.1f}%
  • Overfitting Gap: {avg_overfit:.4f}

🎯 ACCURACY METRICS
  • Baseline (Epoch 1): {df_history['accuracy'].iloc[0]:.2%}
  • Peak: {df_history['accuracy'].max():.2%}
  • Net Improvement: {(df_history['accuracy'].iloc[-1] - df_history['accuracy'].iloc[0]) * 100:.2f}%

⏱️  TEMPORAL GROUNDING
  • Baseline (Epoch 1): {df_history['temporal_iou'].iloc[0]:.4f}
  • Peak: {df_history['temporal_iou'].max():.4f}
  • Net Improvement: {(df_history['temporal_iou'].iloc[-1] - df_history['temporal_iou'].iloc[0]) * 100:.2f}%

💡 RECOMMENDATIONS
"""

# Add recommendations
if avg_overfit > 0.3:
    summary_report += "  ⚠️  High overfitting detected\n"
    summary_report += "     → Consider: L1/L2 regularization, dropout, more data augmentation\n"
if df_history['accuracy'].iloc[-1] < 0.5:
    summary_report += "  ⚠️  Low final accuracy\n"
    summary_report += "     → Consider: lower learning rate, more training epochs, check data quality\n"
if df_history['val_loss'].idxmin() < len(df_history) - 3:
    summary_report += "  ⚠️  Best epoch is early in training\n"
    summary_report += "     → Model may benefit from: lower initial learning rate, longer warmup\n"

summary_report += "\n✅ Training completed successfully!\n"
print(summary_report)

## 7. Model Architecture Efficiency

In [ ]:
# Architecture analysis (conceptual)
architecture_info = {
    'Component': ['Feature Extractor', 'Temporal Binding', 'CHIMRT Reasoning', 'Adaptive Depth', 'Multimodal VDB', 'Answer Generator'],
    'Parameters (M)': [86, 15, 25, 8, 12, 5],
    'Computation Cost': ['Low (frozen)', 'Medium', 'High (multi-hop)', 'Low (gating)', 'Medium (retrieval)', 'Low'],
}

df_arch = pd.DataFrame(architecture_info)
total_params = df_arch['Parameters (M)'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Parameter distribution
colors = plt.cm.Set3(np.linspace(0, 1, len(df_arch)))
axes[0].pie(df_arch['Parameters (M)'], labels=df_arch['Component'], autopct='%1.1f%%', colors=colors)
axes[0].set_title(f'Model Parameters Distribution (Total: {total_params}M)')

# Parameters per component
axes[1].barh(df_arch['Component'], df_arch['Parameters (M)'], color=colors, edgecolor='black')
axes[1].set_xlabel('Parameters (Millions)')
axes[1].set_title('Model Components by Parameter Count')
axes[1].grid(True, axis='x', alpha=0.3)

for i, v in enumerate(df_arch['Parameters (M)']):
    axes[1].text(v + 1, i, f'{v}M', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nModel Architecture Summary:")
print(df_arch.to_string(index=False))
print(f"\nTotal Parameters: {total_params}M")